In [3]:
# ---------------------------------------------------
# API 키를 환경변수로 관리하기 위한 설정 파일
# ---------------------------------------------------
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

In [4]:
# ---------------------------------------------------
# 필수 라이브러리 임포트 및 PDF 로드
# ---------------------------------------------------
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# PDF 로드 (실습 속도를 위해 금융·재정·조세 섹션만 사용)
loader = PyMuPDFLoader("./data/2026_gov.pdf")
docs = loader.load()
docs = docs[41:95]  # 금융·재정·조세 섹션 (54페이지)

print(f"로드된 페이지 수: {len(docs)}")
print(f"첫 페이지 미리보기: {docs[0].page_content[:200]}...")

로드된 페이지 수: 54
첫 페이지 미리보기: 1
2026년부터 이렇게 달라집니다
2026년부터
이렇게 달라집니다
금융·재정·조세01...


In [5]:
# ---------------------------------------------------
# 문서 분할 (청크 생성)
# ---------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # 각 청크 최대 500자
    chunk_overlap=50    # 문맥 보존을 위한 겹침
)
split_docs = text_splitter.split_documents(docs)

print(f"원본 페이지 수: {len(docs)}")
print(f"분할된 청크 수: {len(split_docs)}")
print(f"\n첫 번째 청크:\n{split_docs[0].page_content}")

원본 페이지 수: 54
분할된 청크 수: 88

첫 번째 청크:
1
2026년부터 이렇게 달라집니다
2026년부터
이렇게 달라집니다
금융·재정·조세01


In [6]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(
    documents=split_docs,
    embedding=embeddings
)

In [7]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [8]:
# ---------------------------------------------------
# 기본 RAG 체인 (인용 없음)
# ---------------------------------------------------
basic_prompt = PromptTemplate.from_template(
    """당신은 문서 기반 질의응답 AI 어시스턴트입니다.
주어진 문맥(Context)을 바탕으로 질문에 답변하세요.
문맥에 없는 내용은 "해당 정보를 찾을 수 없습니다"라고 답하세요.

#Context:
{context}

#Question:
{question}

#Answer:"""
)

# 기본 RAG 체인 — LCEL로 연결
basic_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | basic_prompt
    | llm
    | StrOutputParser()
)

print("기본 RAG 체인 생성 완료")

기본 RAG 체인 생성 완료


In [10]:
question="2026년 달라지는 세제 지원 제도에는 어떤 것들이 있나요?"

basic_answer = basic_chain.invoke(question)

print(basic_answer)

# 오리온 프롬프팅을 하기 위해서 청크마다 메타데이터에 uuid를 부여해서 벡터스토어에 저장

2026년 달라지는 세제 지원 제도에는 다음과 같은 내용이 있습니다:

1. **지방이전 기업 세제지원 제도 개선**:
   - 적용대상 및 감면기간 확대
   - 감면한도 신설: 지방투자누계액 × 70% + 지방근무상시근로자 수 × 1,500만원 (청년·서비스업 2,000만원)
   - 사후관리 신설: 감면받은 후 2년 내 상시근로자 수 감소 시 추징 (1명당 1,500만원, 청년·서비스업 2,000만원)
   - 시행일: 2026년 1월 1일 이후 공장ㆍ본사를 이전하는 분부터 적용

2. **고향사랑기부금 세액공제 확대**:
   - 10만원 초과 20만원 이하 고향사랑기부금에 대한 세액공제율을 15%에서 40%로 상향
   - 시행일: 2026년 1월 1일 이후 기부하는 분부터 적용

3. **청년미래적금 신설**:
   - 청년 자산형성 지원을 강화한 청년미래적금이 도입
   - 가입기간을 3년으로 단축하고, 정부기여금 지원비율을 일반형 6%, 우대형 12%로 설정
   - 월 납입한도가 50만원인 자유적립식 비과세 적금상품으로, 최대 납입 시 만기에 2,000만원 이상의 목돈을 수령할 수 있을 것으로 기대됨
   - 시행일: 2026년 6월

이와 같은 제도 개선이 2026년부터 시행됩니다.


In [11]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

def format_docs_with_id(docs: list[Document]) -> str:
    """
        docs: retriever에서 조회된 chunk(문서)
        ['안녕하세요, 반갑습니다.', '제 이름은 땡땡이에요'] -> formatted = []
    """
    formatted = []

    for i, doc in enumerate(docs):
        formatted.append(f"[Source_{i}] {doc.page_content}")
    
    return "\n\n".join(formatted)

sample_docs = retriever.invoke(question)
print(format_docs_with_id(sample_docs))

[Source_0] 23
2026년부터 이렇게 달라집니다
지방이전 기업 세제지원 제도 개선
재정경제부
www.moef.go.kr
지역균형발전을 지원하고 제도를 합리화하기 위하여, 지방이전 기업 세제지원 제도의 적용대상, 
감면기간을 확대하고 감면한도 및 사후관리를 신설합니다.
‌ 	 (대상지역 및 감면기간)
‌ 	 (감면한도) 지방투자누계액 × 70%+지방근무상시근로자 수 × 1,500만원 
     (청년·서비스업 2,000만원)
‌ 	 (사후관리) 감면받은 후 2년 내 상시근로자 수 감소시 추징*
 * 1명당 1,500만원(청년·서비스업 2,000만원)
개정내용은 2026년 1월 1일 이후 공장ㆍ본사를 이전하는 분부터 적용됩니다.
지방이전 기업 세제지원 제도 개선
추진배경
지역균형발전 지원 및 제도 합리화
주요내용
적용대상·감면기간 확대, 감면한도 및 사후관리 신설
시행일
2026년 1월 1일 이후 공장ㆍ본사를 이전하는 분부터 적용

[Source_1] 1
2026년부터 이렇게 달라집니다
2026년부터
이렇게 달라집니다
금융·재정·조세01

[Source_2] 7
2026년부터 이렇게 달라집니다
 
청년미래적금 신설
시행일: 2026년 6월 
자세한 내용은p.051
금융위원회  
11.
• 2026년부터 청년 자산형성 지원을 강화한 청년미래적금이 새롭게 도입됩니다.
• 가입기간을 3년으로 단축해 청년의 장기가입 부담을 줄였으며, 정부기여금 지원비율(일반형-6%, 
우대형 12%)도 높은 수준으로 설정되었습니다.

[Source_3] 31
2026년부터 이렇게 달라집니다
고향사랑기부금 세액공제 확대
재정경제부
www.moef.go.kr
10만원 초과 20만원 이하 고향사랑기부금에 대한 세액공제율을 15→40%로 상향합니다.
* (현행) (~10만원) 100/110, (~2천만원) 15%[특별재난지역 30%]
(개정안) (~10만원) 100/110, (~20만원) 40%<신설> (~2천만원) 15%[상동]
개정내용은 2026년 1월 1일 이후 기부하는

In [12]:
# Orion 시스템 프롬프트 — 인용을 강제하는 CoT 지시
orion_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 검색된 문서로부터 질문의 답변을 작성하는 언어 모델입니다.

### 지시사항
1. 질문에 대해 주어진 다수의 문서(검색 결과)에서 답을 찾아 작성하세요.
2. 검색된 문서에서 답을 찾을 수 없는 경우 "해당 정보를 찾을 수 없습니다"라고 답하세요.
3. 답변 작성 시 먼저 어떤 문서가 관련 있는지 판단한 후 답변을 작성하세요.
4. 답변의 마지막에 반드시 참조한 문서의 Source_id를 [Source_0, Source_2] 형태로 표기하세요.
5. 한국어로 답변하세요."""),
    ("human", """검색 결과:\n{context}\n\n질문: {question}\n답변:""")
])

orion_chain = (
    {"context": retriever | RunnableLambda(format_docs_with_id), "question": RunnablePassthrough()}
    | orion_prompt
    | llm
    | StrOutputParser()
)

answer_orion = orion_chain.invoke(question)
print(f' ==> [Line 21]: \033[38;2;100;201;195m[answer_orion]\033[0m({type(answer_orion).__name__}) = \033[38;2;103;64;139m{answer_orion}\033[0m')


 ==> [Line 21]: [answer_orion](str) = 2026년부터 달라지는 세제 지원 제도에는 다음과 같은 내용이 포함됩니다:

1. **지방이전 기업 세제지원 제도 개선**: 지방으로 이전하는 기업에 대한 세제 지원이 확대됩니다. 적용대상과 감면기간이 늘어나며, 감면한도와 사후관리 규정이 신설됩니다. 감면한도는 지방투자누계액의 70%와 지방근무상시근로자 수에 따라 달라지며, 사후관리는 감면받은 후 2년 내 상시근로자 수가 감소할 경우 추징이 이루어집니다. 이 제도는 2026년 1월 1일 이후 공장이나 본사를 이전하는 기업부터 적용됩니다. [Source_0]

2. **고향사랑기부금 세액공제 확대**: 10만원 초과 20만원 이하의 고향사랑기부금에 대한 세액공제율이 15%에서 40%로 상향 조정됩니다. 이 개정안은 2026년 1월 1일 이후 기부하는 경우에 적용됩니다. [Source_3]

이 외에도 청년미래적금 신설과 같은 청년 자산형성 지원 제도도 도입될 예정입니다. [Source_2, Source_4]

따라서, 2026년에는 지방이전 기업에 대한 세제 지원과 고향사랑기부금 세액공제의 변화가 주요한 세제 지원 제도로 나타납니다. [Source_0, Source_3]


In [13]:
# ---------------------------------------------------
# Cohere 방식 — 인라인 인용 태그 CoT 체인
# ---------------------------------------------------

# Cohere 시스템 프롬프트 — 인라인 태그를 강제하는 CoT 지시
cohere_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 검색된 문서로부터 질문의 답변을 작성하는 언어 모델입니다.

### 지시사항
1. 질문에 대해 주어진 다수의 문서(검색 결과)에서 답을 찾아 작성하세요.
2. 검색된 문서에서 답을 찾을 수 없는 경우 "해당 정보를 찾을 수 없습니다"라고 답하세요.
3. 답변 작성 시 먼저 어떤 문서가 관련 있는지 판단한 후 답변을 작성하세요.
4. 검색된 문서에서 인용한 내용은 반드시 <co: N>인용 내용</co: N> 형태로 감싸세요.
    - N은 해당 문서의 Source_id 번호입니다.
    - 예: <co: 0>세액감면 기준금액이 상향됩니다</co: 0>
5. 인용하지 않은 부분(연결어, 설명 등)은 태그 없이 작성하세요.
6. 한국어로 답변하세요."""),
    ("human", """검색 결과:\n{context}\n\n질문: {question}\n답변:""")
])

# Cohere CoT 체인 — retriever → format_docs_with_id → prompt → llm
cohere_chain = (
    {"context": retriever | RunnableLambda(format_docs_with_id), "question": RunnablePassthrough()}
    | cohere_prompt
    | llm
    | StrOutputParser()
)

print("Cohere CoT 체인 생성 완료")

Cohere CoT 체인 생성 완료


In [15]:
cohere_answer = cohere_chain.invoke(question)
print(f' ==> [Line 1]: \033[38;2;113;57;154m[cohere_answer]\033[0m({type(cohere_answer).__name__}) = \033[38;2;160;153;35m{cohere_answer}\033[0m')



 ==> [Line 1]: [cohere_answer](str) = 2026년부터 달라지는 세제 지원 제도에는 다음과 같은 내용이 포함됩니다.

1. **지방이전 기업 세제지원 제도 개선**: 지방이전 기업에 대한 세제지원의 적용대상과 감면기간이 확대되며, 감면한도와 사후관리도 신설됩니다. 감면한도는 지방투자누계액의 70%와 지방근무상시근로자 수에 따라 달라지며, 청년 및 서비스업에 대해서는 더 높은 한도가 적용됩니다. 이 제도는 2026년 1월 1일 이후 공장이나 본사를 이전하는 기업부터 적용됩니다 <co: 0>지방이전 기업 세제지원 제도의 적용대상, 감면기간을 확대하고 감면한도 및 사후관리를 신설합니다</co: 0>.

2. **고향사랑기부금 세액공제 확대**: 10만원 초과 20만원 이하의 고향사랑기부금에 대한 세액공제율이 15%에서 40%로 상향됩니다. 이 개정안은 2026년 1월 1일 이후 기부하는 분부터 적용됩니다 <co: 3>10만원 초과 20만원 이하 고향사랑기부금에 대한 세액공제율을 15→40%로 상향합니다</co: 3>.

3. **청년미래적금 신설**: 청년 자산형성 지원을 강화한 청년미래적금이 도입됩니다. 가입기간이 3년으로 단축되고, 정부기여금 지원비율이 일반형 6%, 우대형 12%로 설정됩니다. 이 적금은 월 납입한도가 50만원이며, 최대 납입 시 만기에 2,000만원 이상의 목돈을 수령할 수 있을 것으로 기대됩니다. 이 제도는 2026년 6월부터 시행됩니다 <co: 2>청년 자산형성 지원을 강화한 청년미래적금이 새롭게 도입됩니다</co: 2>.

이와 같은 변화들은 지역균형발전과 청년 지원을 목표로 하고 있습니다.
